In [ ]:
!pip install ultralytics roboflow -q
from ultralytics import YOLO
import os

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 56.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="LpOBfSxnkvicemyEuHPG")
project = rf.workspace("images-r8w7j").project("construction-and-demolition-waste")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Construction-and-demolition-waste-1 in yolov8:: 100%|██████████| 372/372 [00:00<00:00, 4966.38it/s]


In [ ]:
import os

# Check the actual folder structure
for root, dirs, files in os.walk("/content/Construction-and-demolition-waste-1"):
    level = root.replace("/content/Construction-and-demolition-waste-1", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

Construction-and-demolition-waste-1/
  train/
    images/
    labels/


In [ ]:
with open("/content/Construction-and-demolition-waste-1/data.yaml", "r") as f:
    print(f.read())

names:
- Construction and demolition waste
nc: 1
roboflow:
  license: CC BY 4.0
  project: construction-and-demolition-waste
  url: https://universe.roboflow.com/images-r8w7j/construction-and-demolition-waste/dataset/1
  version: 1
  workspace: images-r8w7j
test: ../test/images
train: ../train/images
val: ../valid/images



In [ ]:
# --- ENVIRONMENT SETUP & DATA PREPROCESSING (MODEL 1) ---
# Importing required libraries (YOLO, os, shutil, Path) and authenticating Roboflow API.
# Creating validation directories and manually shifting 20% of the training images
# and their corresponding label text files to create a balanced validation split.

In [ ]:
import yaml
import shutil
import os
from pathlib import Path

# Create valid/images and valid/labels folders
os.makedirs("/content/Construction-and-demolition-waste-1/valid/images", exist_ok=True)
os.makedirs("/content/Construction-and-demolition-waste-1/valid/labels", exist_ok=True)

# Take 20% of train images as validation
train_images = list(Path("/content/Construction-and-demolition-waste-1/train/images").glob("*"))
val_count = max(1, len(train_images) // 5)
val_images = train_images[:val_count]

for img_path in val_images:
    # Copy image
    shutil.copy(img_path, "/content/Construction-and-demolition-waste-1/valid/images/")
    # Copy corresponding label
    label_path = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")
    if label_path.exists():
        shutil.copy(label_path, "/content/Construction-and-demolition-waste-1/valid/labels/")

print(f"Validation set created: {val_count} images")

Validation set created: 36 images


In [ ]:
# --- MODEL 1 TRAINING & INFERENCE (C&D WASTE) ---
# Overriding the default data.yaml with absolute paths to avoid directory nesting errors.
# Initializing the YOLOv8-nano model, training for 30 epochs, and running inference
# to generate visual output frames and bounding box coordinate logs.

In [ ]:
# Fix data.yaml with absolute paths
data_yaml = {
    "train": "/content/Construction-and-demolition-waste-1/train/images",
    "val": "/content/Construction-and-demolition-waste-1/valid/images",
    "nc": 1,
    "names": ["Construction and demolition waste"]
}

with open("/content/Construction-and-demolition-waste-1/data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

print("data.yaml fixed!")

# Now retrain
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data="/content/Construction-and-demolition-waste-1/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    name="cdwaste_model",
    patience=10
)

data.yaml fixed!
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Construction-and-demolition-waste-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cdwaste_model, nbs=64, nms=False, opset=None, optimize=False

In [ ]:
from ultralytics import YOLO

model1 = YOLO("/content/runs/detect/cdwaste_model/weights/best.pt")

results = model1.predict(
    source="/content/Construction-and-demolition-waste-1/valid/images",
    save=True,        # saves output frames with boxes drawn
    save_txt=True,    # saves bbox coords per image
    conf=0.25,
    name="cdwaste_output"
)

print("Inference done. Output at /content/runs/detect/cdwaste_output/")


image 1/36 /content/Construction-and-demolition-waste-1/valid/images/206_jpg.rf.b63db91cc1a5a7b94a78c4e5f39567b4.jpg: 640x640 1 Construction and demolition waste, 13.4ms
image 2/36 /content/Construction-and-demolition-waste-1/valid/images/213_jpg.rf.7f55b5ea1570e3b29d395e86248535bf.jpg: 640x640 1 Construction and demolition waste, 73.5ms
image 3/36 /content/Construction-and-demolition-waste-1/valid/images/220_jpg.rf.9dd97aa8302d239d27093f7c60239026.jpg: 640x640 1 Construction and demolition waste, 34.6ms
image 4/36 /content/Construction-and-demolition-waste-1/valid/images/227_jpg.rf.5de7a10c0d71ffd839176b7299566425.jpg: 640x640 1 Construction and demolition waste, 75.7ms
image 5/36 /content/Construction-and-demolition-waste-1/valid/images/237_jpg.rf.153e6c058351bea735dc22e2f5dcf9f7.jpg: 640x640 2 Construction and demolition wastes, 91.0ms
image 6/36 /content/Construction-and-demolition-waste-1/valid/images/248_jpg.rf.9e7ce2d5014babd893671d64c1f90bd9.jpg: 640x640 1 Construction and dem

In [ ]:
# --- KAGGLE API SETUP & YOLO FORMAT CONVERSION (MODEL 2) ---
# Configuring hidden Kaggle directory for authentication and pulling the Classification dataset.
# Iterating through classification folders and programmatically generating standard
# YOLO normalized bounding box text files (x_center, y_center, width, height) for 7 garbage classes.

In [ ]:
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d sumn2u/garbage-classification-v2

!unzip -q garbage-classification-v2.zip -d garbage_v2_dataset

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2
License(s): MIT
100% 1.07G/1.07G [00:16<00:00, 69.0MB/s]



In [ ]:
import os

dataset_path = "/content/garbage_v2_dataset"

print(f"Checking structure of: {dataset_path}\n")

for root, dirs, files in os.walk(dataset_path):
    # Calculate how deep we are in the folder tree
    level = root.replace(dataset_path, "").count(os.sep)

    # Only print top-level folders to keep it clean
    if level < 3:
        indent = " " * 4 * level
        folder_name = os.path.basename(root)

        # Print the folder name
        if folder_name != "":
            print(f"{indent} {folder_name}/")

        # If it's a category folder (no sub-folders inside), print the image count
        if len(dirs) == 0 and len(files) > 0:
            print(f"{indent}    └─ {len(files)} images")

Checking structure of: /content/garbage_v2_dataset

 garbage_v2_dataset/
     original/
         cardboard/
            └─ 1411 images
         glass/
            └─ 1736 images
         trash/
            └─ 453 images
         paper/
            └─ 1336 images
         clothes/
            └─ 1892 images
         plastic/
            └─ 1597 images
         metal/
            └─ 930 images
         battery/
            └─ 756 images
         biological/
            └─ 699 images
         shoes/
            └─ 1449 images
     standardized_256/
         cardboard/
            └─ 1411 images
         glass/
            └─ 1736 images
         trash/
            └─ 453 images
         paper/
            └─ 1336 images
         clothes/
            └─ 1892 images
         plastic/
            └─ 1597 images
         metal/
            └─ 930 images
         battery/
            └─ 756 images
         biological/
            └─ 699 images
         shoes/
            └─ 1449 images
     st

In [ ]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

# Classes relevant to "Garbage on Road" - skip shoes, clothes, battery
CLASSES = ["cardboard", "glass", "trash", "paper", "plastic", "metal", "biological"]
source_dir = "/content/garbage_v2_dataset/standardized_256"
output_dir = "/content/garbage_detection"

for split in ["train", "valid", "test"]:
    os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)

for class_id, class_name in enumerate(CLASSES):
    class_path = Path(source_dir) / class_name
    if not class_path.exists():
        print(f"Skipping {class_name}, folder not found")
        continue

    images = list(class_path.glob("*.jpg")) + list(class_path.glob("*.png")) + list(class_path.glob("*.JPG"))
    train_imgs, temp = train_test_split(images, test_size=0.2, random_state=42)
    val_imgs, test_imgs = train_test_split(temp, test_size=0.5, random_state=42)

    for split_name, split_imgs in [("train", train_imgs), ("valid", val_imgs), ("test", test_imgs)]:
        for img_path in split_imgs:
            # Copy image
            shutil.copy(img_path, f"{output_dir}/{split_name}/images/{class_name}_{img_path.name}")
            # Create full-image bounding box label (x_center, y_center, width, height) normalized
            label_file = f"{output_dir}/{split_name}/labels/{class_name}_{img_path.stem}.txt"
            with open(label_file, "w") as f:
                f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")

print("Conversion done!")

Conversion done!


In [ ]:
# --- MODEL 2 TRAINING & INFERENCE (GARBAGE ON ROAD) ---
# Dynamically generating data.yaml for the newly formatted Kaggle garbage dataset.
# Training Model 2 and running predictions to save output visual artifacts and text logs.

In [ ]:
import yaml

data = {
    "train": f"{output_dir}/train/images",
    "val": f"{output_dir}/valid/images",
    "test": f"{output_dir}/test/images",
    "nc": len(CLASSES),
    "names": CLASSES
}

with open(f"{output_dir}/data.yaml", "w") as f:
    yaml.dump(data, f)

print("data.yaml created!")
print(yaml.dump(data))

data.yaml created!
names:
- cardboard
- glass
- trash
- paper
- plastic
- metal
- biological
nc: 7
test: /content/garbage_detection/test/images
train: /content/garbage_detection/train/images
val: /content/garbage_detection/valid/images



In [ ]:
from ultralytics import YOLO

model2 = YOLO("yolov8n.pt")
results = model2.train(
    data=f"{output_dir}/data.yaml",
    epochs=25,
    imgsz=256,
    batch=32,
    name="garbage_model",
    patience=10
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/garbage_detection/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=garbage_model-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

In [ ]:
from ultralytics import YOLO

model2 = YOLO("/content/runs/detect/garbage_model-2/weights/best.pt")

results = model2.predict(
    source="/content/garbage_detection/test/images",
    save=True,
    save_txt=True,
    conf=0.25,
    name="garbage_output"
)
print("Done. Output at /content/runs/detect/garbage_output/")


image 1/819 /content/garbage_detection/test/images/biological_biological_118.jpg: 256x256 1 biological, 18.5ms
image 2/819 /content/garbage_detection/test/images/biological_biological_120.jpg: 256x256 1 biological, 11.7ms
image 3/819 /content/garbage_detection/test/images/biological_biological_14.jpg: 256x256 1 biological, 11.5ms
image 4/819 /content/garbage_detection/test/images/biological_biological_143.jpg: 256x256 1 biological, 18.2ms
image 5/819 /content/garbage_detection/test/images/biological_biological_159.jpg: 256x256 1 biological, 10.5ms
image 6/819 /content/garbage_detection/test/images/biological_biological_172.jpg: 256x256 1 biological, 10.3ms
image 7/819 /content/garbage_detection/test/images/biological_biological_179.jpg: 256x256 1 biological, 8.8ms
image 8/819 /content/garbage_detection/test/images/biological_biological_187.jpg: 256x256 1 biological, 8.0ms
image 9/819 /content/garbage_detection/test/images/biological_biological_188.jpg: 256x256 1 biological, 11.0ms
ima

In [ ]:
# --- MODEL 3 PIPELINE & EVALUATION (OVERFLOWING DUSTBINS) ---
# Downloading the waste-bin fill level dataset via Roboflow, updating absolute paths
# in the configuration yaml, and executing YOLOv8 training/inference to detect spillover conditions.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="LpOBfSxnkvicemyEuHPG")
project = rf.workspace("finalyearproject-vvqft").project("waste-bin-fill-level-detect2-lxsf4")
version = project.version(9)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to waste-bin-fill-level-detect2-9 in yolov8:: 100%|██████████| 2258/2258 [00:00<00:00, 5058.30it/s]


In [ ]:
import glob
yamls = glob.glob("/content/*/data.yaml")
print(yamls)
for y in yamls:
    if "garbage" not in y.lower() or "overflow" in y.lower() or "dustbin" in y.lower():
        with open(y) as f:
            print(y)
            print(f.read())

['/content/garbage_detection/data.yaml', '/content/waste-bin-fill-level-detect2-9/data.yaml', '/content/Construction-and-demolition-waste-1/data.yaml']
/content/waste-bin-fill-level-detect2-9/data.yaml
names:
- empty
- full
- half-full
- overflowing
nc: 4
roboflow:
  license: CC BY 4.0
  project: waste-bin-fill-level-detect2-lxsf4
  url: https://universe.roboflow.com/finalyearproject-vvqft/waste-bin-fill-level-detect2-lxsf4/dataset/9
  version: 9
  workspace: finalyearproject-vvqft
test: ../test/images
train: ../train/images
val: ../valid/images

/content/Construction-and-demolition-waste-1/data.yaml
names:
- Construction and demolition waste
nc: 1
train: /content/Construction-and-demolition-waste-1/train/images
val: /content/Construction-and-demolition-waste-1/valid/images



In [ ]:
import yaml
from ultralytics import YOLO

# Point ONLY to the folder directory
dataset_path = "/content/waste-bin-fill-level-detect2-9"

# Fix data.yaml with absolute paths
data = {
    "train": f"{dataset_path}/train/images",
    "val": f"{dataset_path}/valid/images",
    "test": f"{dataset_path}/test/images",
    "nc": 4,
    "names": ["empty", "full", "half-full", "overflowing"]
}

with open(f"{dataset_path}/data.yaml", "w") as f:
    yaml.dump(data, f)

print("data.yaml fixed!")

# Train Model 3
model3 = YOLO("yolov8n.pt")
results = model3.train(
    data=f"{dataset_path}/data.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    name="dustbin_model",
    patience=10
)

data.yaml fixed!
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/waste-bin-fill-level-detect2-9/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dustbin_model, nbs=64, nms=False, opset=None, optimize=False, opt

In [ ]:
from ultralytics import YOLO

model3 = YOLO("/content/runs/detect/dustbin_model/weights/best.pt")
results = model3.predict(
    source="/content/waste-bin-fill-level-detect2-9/test/images",
    save=True,
    save_txt=True,
    conf=0.25,
    name="dustbin_output"
)
print("Done.")


image 1/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG-20251016-WA0015_jpg.rf.f2007cb7f689ea62f21c3f6bfe98ec14.jpg: 640x640 2 fulls, 7.8ms
image 2/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG-20251016-WA0017_jpg.rf.c52fa0c906576e130fe07aeec31c1a76.jpg: 640x640 1 full, 7.2ms
image 3/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG_20251014_164036_jpg.rf.533b8df14a0084b218bc6a69c85d1a6c.jpg: 640x640 1 half-full, 7.2ms
image 4/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG_20251014_164200_jpg.rf.51a530b543840e30ea23209b96a4b8dc.jpg: 640x640 1 half-full, 7.3ms
image 5/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG_20251014_164236_jpg.rf.f7d23414eb7261853330f80c4e42a3d3.jpg: 640x640 1 empty, 1 half-full, 7.2ms
image 6/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG_20251014_165246_jpg.rf.f171538bc0e4f2599d28c2d0ec7e108b.jpg: 640x640 1 full, 1 half-full, 7.2ms
image 7/44 /content/waste-bin-fill-level-detect2-9/test/image

In [ ]:
# --- ARTIFACT PACKAGING & EXPORT ---
# Compressing the output directories containing visual frames and text coordinates,
# and triggering local downloads to package the final submission payload.

In [ ]:
import shutil
from google.colab import files

print("Zipping Model 1 files...")
shutil.make_archive("/content/CD_Waste_Outputs", 'zip', "/content/runs/detect/cdwaste_output")
files.download("/content/CD_Waste_Outputs.zip")
files.download("/content/runs/detect/cdwaste_model/weights/best.pt")

print("Zipping Model 2 files...")
shutil.make_archive("/content/Garbage_Outputs", 'zip', "/content/runs/detect/garbage_output")
files.download("/content/Garbage_Outputs.zip")
files.download("/content/runs/detect/garbage_model-2/weights/best.pt")

Zipping Model 1 files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Zipping Model 2 files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

print("Zipping Model 3 files...")
# Zip the output images and coordinates
shutil.make_archive("/content/Dustbin_Outputs", 'zip', "/content/runs/detect/dustbin_output")

# Trigger the downloads
files.download("/content/Dustbin_Outputs.zip")
files.download("/content/runs/detect/dustbin_model/weights/best.pt")

Zipping Model 3 files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

# Zip the raw input test images
shutil.make_archive("/content/Input_Test_Images", 'zip', "/content/garbage_detection/test/images")

# Download the zip
files.download("/content/Input_Test_Images.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

print("Zipping Model 1 Input Images...")
# We used the validation set for Model 1 inference
shutil.make_archive("/content/Model1_Inputs", 'zip', "/content/Construction-and-demolition-waste-1/valid/images")
files.download("/content/Model1_Inputs.zip")

print("Zipping Model 3 Input Images...")
# We used the test set for Model 3 inference
shutil.make_archive("/content/Model3_Inputs", 'zip', "/content/waste-bin-fill-level-detect2-9/test/images")
files.download("/content/Model3_Inputs.zip")

Zipping Model 1 Input Images...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Zipping Model 3 Input Images...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:

# MODEL 3 - Overflowing Dustbin Detection (Fine-tuning Round 2)
# Dataset: Waste Bin Fill Level Detect2 (Roboflow)
# Classes: empty, full, half-full, overflowing
#
# Round 1 (20 epochs, lr=0.01) gave mAP50=0.808, below threshold.
# Round 2 starts from Round 1 weights with lower lr and augmentation
# to improve convergence on the small dataset.
# Final mAP50 = 0.877

In [ ]:
from ultralytics import YOLO

# Start from your existing best weights
model = YOLO("/content/runs/detect/dustbin_model/weights/best.pt")

model.train(
    data="/content/waste-bin-fill-level-detect2-9/data.yaml",
    epochs=30,        # more epochs
    imgsz=640,
    batch=8,          # smaller batch helps with small dataset
    lr0=0.001,        # lower learning rate for fine-tuning
    name="dustbin_model_v2",
    patience=15
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/waste-bin-fill-level-detect2-9/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/dustbin_model/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dustbin_model_v2, nbs=64, nms=False, opset

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78149785acc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
import shutil
from google.colab import files

# Zip the raw input test images
shutil.make_archive("/content/Model3v2_Inputs", 'zip', "/content/waste-bin-fill-level-detect2-9/test/images")

# Download the zip
files.download("/content/Model3v2_Inputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/dustbin_model_v2/weights/best.pt")
results = model.predict(
    source="/content/waste-bin-fill-level-detect2-9/test/images",
    save=True,
    save_txt=True,
    conf=0.25,
    name="dustbin_output_v2"
)
print("Done.")


image 1/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG-20251016-WA0015_jpg.rf.f2007cb7f689ea62f21c3f6bfe98ec14.jpg: 640x640 2 fulls, 28.4ms
image 2/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG-20251016-WA0017_jpg.rf.c52fa0c906576e130fe07aeec31c1a76.jpg: 640x640 2 fulls, 17.5ms
image 3/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG_20251014_164036_jpg.rf.533b8df14a0084b218bc6a69c85d1a6c.jpg: 640x640 1 half-full, 23.8ms
image 4/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG_20251014_164200_jpg.rf.51a530b543840e30ea23209b96a4b8dc.jpg: 640x640 1 half-full, 11.2ms
image 5/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG_20251014_164236_jpg.rf.f7d23414eb7261853330f80c4e42a3d3.jpg: 640x640 1 empty, 2 half-fulls, 12.0ms
image 6/44 /content/waste-bin-fill-level-detect2-9/test/images/IMG_20251014_165246_jpg.rf.f171538bc0e4f2599d28c2d0ec7e108b.jpg: 640x640 1 full, 1 overflowing, 19.4ms
image 7/44 /content/waste-bin-fill-level-detect2-9/

In [ ]:
import shutil

shutil.copytree(
    "/content/runs/detect/dustbin_output_v2",
    "/content/submission/model3_dustbin/output_frames_v2",
    dirs_exist_ok=True
)

# Zip just the dustbin folder
shutil.make_archive("/content/Dustbin_Updated", "zip", "/content/submission/model3_dustbin")
print("Ready at /content/Dustbin_Updated.zip")

from google.colab import files
files.download("/content/Dustbin_Updated.zip")

Ready at /content/Dustbin_Updated.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
readme = """# CityLens Hackathon 2026 - Group 51
## KPI Group 1: Cleanliness and Waste Management

Submission by Group 51 for the CityLens AI and Computer Vision Hackathon 2026,
organized by ARF and CDIS Tech Teams.

---

## Problem Statement

Detect civic cleanliness and waste management issues from CCTV images and video
frames with a minimum accuracy of 85% across all assigned categories.

---

## Models

| # | Category | Weights File | mAP50 | Priority Weight |
|---|----------|-------------|-------|-----------------|
| 1 | Construction and Demolition Waste | best-4_model1.pt | 0.961 | 50% |
| 2 | Garbage on Road | best-4model2.pt | 0.943 | 30% |
| 3 | Overflowing Dustbins | best-6.pt | 0.877 | 20% |

Weighted average mAP50: 0.934

---

## Architecture

All three models use YOLOv8n (nano variant) from Ultralytics, fine-tuned on
domain-specific datasets. YOLOv8n was chosen for its balance of speed and
accuracy, making it deployable in real-time CCTV inference scenarios.

- Framework: PyTorch 2.11.0
- Ultralytics version: 8.4.75
- GPU: Tesla T4 (Google Colab)
- Input resolution: 640x640 (Models 1 and 3), 256x256 (Model 2)

---

## Datasets

### Model 1 - Construction and Demolition Waste
- Dataset: construction-solid-waste
- Source: Roboflow Universe
- Size: 183 images
- License: CC BY 4.0
- URL: https://universe.roboflow.com/images-r8w7j/construction-solid-waste

### Model 2 - Garbage on Road
- Dataset: Garbage Classification v2
- Source: Kaggle (sumn2u)
- Size: 13,000 images across 7 classes
- Classes: cardboard, glass, trash, paper, plastic, metal, biological
- URL: https://kaggle.com/datasets/sumn2u/garbage-classification-v2
- Note: Dataset was in image classification format. Converted to YOLO detection
  format with full-image bounding boxes assigned per class label.

### Model 3 - Overflowing Dustbins
- Dataset: Waste Bin Fill Level Detect2
- Source: Roboflow Universe (finalyearproject-vvqft)
- Size: 460 images
- Classes: empty, full, half-full, overflowing
- License: CC BY 4.0
- URL: https://universe.roboflow.com/finalyearproject-vvqft/waste-bin-fill-level-detect2-lxsf4

---

## Training Details

### Model 1 - C&D Waste
- Base weights: yolov8n.pt (COCO pretrained)
- Epochs: 30
- Batch size: 16
- Image size: 640
- Final mAP50: 0.961

### Model 2 - Garbage on Road
- Base weights: yolov8n.pt (COCO pretrained)
- Epochs: 25
- Batch size: 32
- Image size: 256
- Final mAP50: 0.943

### Model 3 - Overflowing Dustbins
- Base weights: yolov8n.pt, trained in two rounds
- Round 1: 20 epochs, batch=16, lr=0.01, mAP50=0.808
- Round 2: 30 epochs, batch=8, lr=0.001, augment=True (fine-tuned from Round 1)
- Image size: 640
- Final mAP50: 0.877

---

## Output Format

Each label file contains one detection per line:
class_id x_center y_center width height

All values are normalized between 0 and 1 relative to image dimensions.

To recover pixel coordinates:
- x_pixel = x_center * image_width
- y_pixel = y_center * image_height
- w_pixel = width * image_width
- h_pixel = height * image_height

---

## Per-Class Results

### Model 1 - C&D Waste
| Class | mAP50 |
|-------|-------|
| Construction and demolition waste | 0.961 |

### Model 2 - Garbage on Road
| Class | mAP50 |
|-------|-------|
| cardboard | 0.979 |
| glass | 0.972 |
| trash | 0.830 |
| paper | 0.948 |
| plastic | 0.942 |
| metal | 0.939 |
| biological | 0.993 |

### Model 3 - Overflowing Dustbins
| Class | mAP50 |
|-------|-------|
| empty | 0.995 |
| full | 0.771 |
| half-full | 0.956 |
| overflowing | 0.786 |

---

## External References

- Ultralytics YOLOv8: https://github.com/ultralytics/ultralytics
- Roboflow Universe: https://universe.roboflow.com
- Kaggle Datasets: https://kaggle.com
- PyTorch: https://pytorch.org

---

## LLMs Used

Claude (Anthropic) was used for guidance on training pipeline setup,
dataset format conversion, yaml fixes, and debugging during the hackathon.
"""

with open("/content/README.md", "w") as f:
    f.write(readme)

print("README written.")

from google.colab import files
files.download("/content/README.md")


README written.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>